# Tratamento dos Dados — Municípios e Regionais de Saúde

---
## 1. Imports

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

---
## 2. Configurações Gerais


In [2]:
# ── Caminhos de entrada ───────────────────────────────────────────────────────
ENTRADA_MUNICIPIOS  = Path("/home/anna/tcc/Variáveis/Municípios/Relacao-de-municipios-por-vinculacao-administrativa-090.xlsx")
ENTRADA_MORTALIDADE = Path("/home/anna/tcc/Seleção e Transformação/Mortalidade/tabela_mortalidade_faixa_etaria.csv")

# ── Caminho de saída ──────────────────────────────────────────────────────────
SAIDA = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

# ── Parâmetros de leitura do Excel ────────────────────────────────────────────
ABA_EXCEL  = "RegionalizaçãoMG2024"   # nome da aba da planilha SES-MG
HEADER_ROW = 3                         # linha do cabeçalho (0-indexado)

# ── Nomes das colunas no CSV de mortalidade ───────────────────────────────────
COL_MUNICIPIO_MORT = "Municípios"
COL_REGIONAL_MORT  = "Regional de Saúde"

# ── Correções de grafia: nome_no_excel → nome_na_mortalidade ─────────────────
# Adicione aqui qualquer município com grafia divergente entre as duas fontes.
CORRECOES_NOMES = {
    "brazopolis"           : "brasopolis",
    "dona euzebia"         : "dona eusebia",
    "sao tome das letras"  : "sao thome das letras",
}

SAIDA.parent.mkdir(parents=True, exist_ok=True)
print(f"✅ Pasta de saída: {SAIDA.parent}")

✅ Pasta de saída: /home/anna/tcc/Seleção e Transformação/Municípios


---
## 3. Funções Auxiliares

`normalizar_texto` é a única função utilitária necessária neste pipeline —
remove acentos e converte para minúsculas para permitir o join entre as duas fontes.

In [ ]:
def normalizar_texto(texto) -> str:
    """Remove acentos e converte para minúsculas. Trata NaN como string vazia."""
    texto = "" if pd.isna(texto) else str(texto).strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower()



✅ Funções auxiliares definidas.


---
## 4. Carregamento dos Dados

### 4.1 Municípios — planilha SES-MG

Lê a aba `RegionalizaçãoMG2024` do Excel, extraindo código IBGE e nome do município.

In [4]:
df_excel = pd.read_excel(ENTRADA_MUNICIPIOS, sheet_name=ABA_EXCEL, header=HEADER_ROW)

tabela = pd.DataFrame({
    "codigo_ibge"           : df_excel["Código IBGE"],
    "municipio_original"    : df_excel["Município "].astype(str).str.strip(),
    "municipio_normalizado" : df_excel["Município "].map(normalizar_texto),
})

# Aplica correções de grafia antes do merge
tabela["municipio_normalizado"] = tabela["municipio_normalizado"].replace(CORRECOES_NOMES)

print(f"✅ Excel carregado: {len(tabela)} municípios")
display(tabela.head(3))

✅ Excel carregado: 853 municípios


,codigo_ibge,municipio_original,municipio_normalizado
0,3100104,Abadia dos Dourados,abadia dos dourados
1,3100203,Abaeté,abaete
2,3100302,Abre Campo,abre campo


### 4.2 Regionais de Saúde — CSV de mortalidade

In [5]:
mort = pd.read_csv(ENTRADA_MORTALIDADE, encoding="utf-8-sig")

regionais = (
    mort[[COL_MUNICIPIO_MORT, COL_REGIONAL_MORT]]
    .drop_duplicates(subset=COL_MUNICIPIO_MORT)
    .assign(municipio_normalizado=lambda d: d[COL_MUNICIPIO_MORT].map(normalizar_texto))
    .rename(columns={COL_REGIONAL_MORT: "regional_original"})
    [["municipio_normalizado", "regional_original"]]
)

print(f"✅ CSV de mortalidade carregado: {len(regionais)} municípios únicos com regional")
display(regionais.head(3))

✅ CSV de mortalidade carregado: 841 municípios únicos com regional


,municipio_normalizado,regional_original
0,alfenas,Alfenas
1,alterosa,Alfenas
2,arceburgo,Alfenas


---
## 5. Análise Inicial

Verificação rápida das duas fontes antes do join: dimensões e valores ausentes.

In [6]:
display(Markdown(f"**Municípios no Excel:** {len(tabela)} | **Municípios com regional:** {len(regionais)}"))

# Municípios no Excel que não estão no CSV de mortalidade (serão NaN após o join)
chaves_excel = set(tabela["municipio_normalizado"].dropna())
chaves_mort  = set(regionais["municipio_normalizado"].dropna())
sem_regional_preview = chaves_excel - chaves_mort

display(Markdown(f"**Municípios sem correspondência no CSV de mortalidade (prévia):** {len(sem_regional_preview)}"))
if sem_regional_preview:
    for m in sorted(sem_regional_preview)[:10]:
        print(f"  - {m}")
    if len(sem_regional_preview) > 10:
        print(f"  ... e mais {len(sem_regional_preview) - 10}")

**Municípios no Excel:** 853 | **Municípios com regional:** 841

**Municípios sem correspondência no CSV de mortalidade (prévia):** 13

  - berizal
  - cajuri
  - claraval
  - consolacao
  - divisa alegre
  - dores de campos
  - douradoquara
  - grupiara
  - monte formoso
  - passa vinte
  ... e mais 3


---
## 6. Join pelo Nome Normalizado

Une a tabela de municípios com as regionais de saúde usando `municipio_normalizado` como chave.
Adiciona também `regional_normalizada` para facilitar joins futuros.

In [7]:
tabela = tabela.merge(regionais, on="municipio_normalizado", how="left")
tabela["regional_normalizada"] = tabela["regional_original"].map(normalizar_texto)

print(f"✅ Join concluído. Shape: {tabela.shape}")
display(tabela.head(5))

✅ Join concluído. Shape: (853, 5)


,codigo_ibge,municipio_original,municipio_normalizado,regional_original,regional_normalizada
0,3100104,Abadia dos Dourados,abadia dos dourados,Uberlândia,uberlandia
1,3100203,Abaeté,abaete,Sete Lagoas,sete lagoas
2,3100302,Abre Campo,abre campo,Manhumirim,manhumirim
3,3100401,Acaiaca,acaiaca,Ponte Nova,ponte nova
4,3100500,Açucena,acucena,Coronel Fabriciano,coronel fabriciano


---
## 7. Validação Pós-Join

Verifica municípios sem regional associada e exibe um relatório completo.

In [8]:
sem_regional = tabela["regional_original"].isna().sum()
com_regional = tabela["regional_original"].notna().sum()

print("╔══════════════════════════════════════════════════╗")
print("║              Resultado do join                   ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Total de municípios        : {len(tabela):<18} ║")
print(f"║  Com regional identificada  : {com_regional:<18} ║")
print(f"║  Sem regional (NaN)         : {sem_regional:<18} ║")
print("╚══════════════════════════════════════════════════╝")

if sem_regional:
    display(Markdown(f"⚠️ **{sem_regional} município(s) sem regional — ausentes na base de mortalidade:**"))
    display(
        tabela[tabela["regional_original"].isna()][["codigo_ibge", "municipio_original"]]
        .reset_index(drop=True)
        .style.hide(axis="index")
    )
else:
    display(Markdown("✅ **Todos os municípios tiveram regional identificada.**"))

╔══════════════════════════════════════════════════╗
║              Resultado do join                   ║
╠══════════════════════════════════════════════════╣
║  Total de municípios        : 853                ║
║  Com regional identificada  : 840                ║
║  Sem regional (NaN)         : 13                 ║
╚══════════════════════════════════════════════════╝


⚠️ **13 município(s) sem regional — ausentes na base de mortalidade:**

codigo_ibge,municipio_original
3106655,Berizal
3110202,Cajuri
3116407,Claraval
3118502,Consolação
3122355,Divisa Alegre
3123007,Dores de Campos
3123502,Douradoquara
3127909,Grupiara
3143153,Monte Formoso
3147808,Passa Vinte


---
## 8. Salvando o Resultado

In [9]:
tabela.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("╔══════════════════════════════════════════════════════╗")
print("║              Arquivo gerado com sucesso              ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Arquivo : {str(SAIDA.name):<42}║")
print(f"║  Linhas  : {len(tabela):<42}║")
print(f"║  Colunas : {list(tabela.columns)}")
print("╚══════════════════════════════════════════════════════╝")

display(Markdown("**Amostra da tabela final:**"))
display(tabela.head(5))

╔══════════════════════════════════════════════════════╗
║              Arquivo gerado com sucesso              ║
╠══════════════════════════════════════════════════════╣
║  Arquivo : tabela_municipios_regionais.csv           ║
║  Linhas  : 853                                       ║
║  Colunas : ['codigo_ibge', 'municipio_original', 'municipio_normalizado', 'regional_original', 'regional_normalizada']
╚══════════════════════════════════════════════════════╝


**Amostra da tabela final:**

,codigo_ibge,municipio_original,municipio_normalizado,regional_original,regional_normalizada
0,3100104,Abadia dos Dourados,abadia dos dourados,Uberlândia,uberlandia
1,3100203,Abaeté,abaete,Sete Lagoas,sete lagoas
2,3100302,Abre Campo,abre campo,Manhumirim,manhumirim
3,3100401,Acaiaca,acaiaca,Ponte Nova,ponte nova
4,3100500,Açucena,acucena,Coronel Fabriciano,coronel fabriciano
